In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
# loading  CICIDS
df = pd.read_csv(
    '../data/raw/cicids/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    low_memory=False
)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f"Shape: {df.shape}")
print(f"\nRaw label distribution:")
print(df['label'].value_counts())

# Fixing  encoding artifact in label names
df['label'] = df['label'].str.encode('latin-1', errors='replace').str.decode('utf-8', errors='replace')
df['label'] = df['label'].str.strip()

# Normalise to clean names
def normalise_label(lbl):
    lbl = str(lbl).strip()
    if lbl == 'BENIGN':
        return 'BENIGN'
    elif 'Brute' in lbl or 'brute' in lbl:
        return 'Web Attack - Brute Force'
    elif 'XSS' in lbl or 'xss' in lbl:
        return 'Web Attack - XSS'
    elif 'Sql' in lbl or 'SQL' in lbl or 'sql' in lbl:
        return 'Web Attack - SQL Injection'
    else:
        return lbl

df['label_clean'] = df['label'].apply(normalise_label)

print(f"\nCleaned label distribution:")
print(df['label_clean'].value_counts())


Shape: (170366, 79)

Raw label distribution:
label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Cleaned label distribution:
label_clean
BENIGN                        168186
Web Attack - Brute Force        1507
Web Attack - XSS                 652
Web Attack - SQL Injection        21
Name: count, dtype: int64


In [4]:
# Binary encoding — has the same  convention as CSIC 2010 (0 = Normal, 1 = Attack)
df['label_binary'] = df['label_clean'].apply(
    lambda x: 0 if x == 'BENIGN' else 1
)

print("Binary label distribution:")
print(df['label_binary'].value_counts())
print(f"\n  Normal (0): {(df['label_binary']==0).sum():,}")
print(f"  Attack (1): {(df['label_binary']==1).sum():,}")
print(f"\n  Attack rate: {df['label_binary'].mean()*100:.2f}%")
print(f"  Imbalance ratio: {(df['label_binary']==0).sum() / (df['label_binary']==1).sum():.1f}:1")

Binary label distribution:
label_binary
0    168186
1      2180
Name: count, dtype: int64

  Normal (0): 168,186
  Attack (1): 2,180

  Attack rate: 1.28%
  Imbalance ratio: 77.1:1


In [5]:
# Replacing  inf/-inf with NaN, then filling  with median
numeric_cols = df.select_dtypes(include=[np.number]).columns

print("Checking for Inf values...")
inf_counts = {}
for col in numeric_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf

print(f"Columns with Inf values: {len(inf_counts)}")
for col, cnt in inf_counts.items():
    print(f"  {col}: {cnt} Inf values")

# Replacing Inf with NaN
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Fill NaN with columns median
nan_counts = df[numeric_cols].isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"\nColumns with NaN after Inf replacement: {len(nan_cols)}")

for col in nan_cols.index:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(f"\nFilled all NaN with column medians")
print(f"Remaining nulls: {df[numeric_cols].isnull().sum().sum()}")
print(f"Dataset shape still: {df.shape}")

Checking for Inf values...
Columns with Inf values: 2
  flow_bytes/s: 115 Inf values
  flow_packets/s: 135 Inf values

Columns with NaN after Inf replacement: 2

Filled all NaN with column medians
Remaining nulls: 0
Dataset shape still: (170366, 81)


In [6]:
SELECTED_FEATURES = [
    'destination_port',
    'flow_duration',
    'total_fwd_packets',
    'total_backward_packets',
    'total_length_of_fwd_packets',
    'total_length_of_bwd_packets',
    'fwd_packet_length_max',
    'fwd_packet_length_min',
    'fwd_packet_length_mean',
    'fwd_packet_length_std',
    'bwd_packet_length_max',
    'bwd_packet_length_min',
    'bwd_packet_length_mean',
    'bwd_packet_length_std',
    'flow_bytes/s',
    'flow_packets/s',
    'psh_flag_count',
    'ack_flag_count',
    'syn_flag_count',
    'rst_flag_count',
    'fin_flag_count',
    'init_win_bytes_forward',
    'init_win_bytes_backward',
    'active_mean',
    'active_std',
    'idle_mean',
    'idle_std',
    'min_packet_length',
    'max_packet_length',
    'packet_length_mean',
    'packet_length_std',
    'packet_length_variance',
]

# Verifying all exist in datasets
missing = [f for f in SELECTED_FEATURES if f not in df.columns]
available = [f for f in SELECTED_FEATURES if f in df.columns]

print(f"Features requested: {len(SELECTED_FEATURES)}")
print(f"Features available: {len(available)}")
print(f"Features missing:   {len(missing)}")
if missing:
    print(f"  Missing: {missing}")

# Build feature matrix
X_cicids = df[available].copy()
y_cicids = df['label_binary'].copy()
y_cicids_multiclass = df['label_clean'].copy()

print(f"\nFeature matrix shape: {X_cicids.shape}")
print(f"Label vector shape:   {y_cicids.shape}")
print(f"\nFeature list:")
for i, f in enumerate(available, 1):
    print(f"  {i:2}. {f}")

Features requested: 32
Features available: 32
Features missing:   0

Feature matrix shape: (170366, 32)
Label vector shape:   (170366,)

Feature list:
   1. destination_port
   2. flow_duration
   3. total_fwd_packets
   4. total_backward_packets
   5. total_length_of_fwd_packets
   6. total_length_of_bwd_packets
   7. fwd_packet_length_max
   8. fwd_packet_length_min
   9. fwd_packet_length_mean
  10. fwd_packet_length_std
  11. bwd_packet_length_max
  12. bwd_packet_length_min
  13. bwd_packet_length_mean
  14. bwd_packet_length_std
  15. flow_bytes/s
  16. flow_packets/s
  17. psh_flag_count
  18. ack_flag_count
  19. syn_flag_count
  20. rst_flag_count
  21. fin_flag_count
  22. init_win_bytes_forward
  23. init_win_bytes_backward
  24. active_mean
  25. active_std
  26. idle_mean
  27. idle_std
  28. min_packet_length
  29. max_packet_length
  30. packet_length_mean
  31. packet_length_std
  32. packet_length_variance


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train/test split — stratified to preserve 77:1 imbalance ratio
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cicids, y_cicids,
    test_size=0.2,
    random_state=42,
    stratify=y_cicids
)

print(f"Train set: {X_train_c.shape}")
print(f"Test set:  {X_test_c.shape}")
print(f"\nTrain label distribution:")
print(y_train_c.value_counts())
print(f"\nTest label distribution:")
print(y_test_c.value_counts())
print(f"\nTrain attack rate: {y_train_c.mean()*100:.2f}%")
print(f"Test attack rate:  {y_test_c.mean()*100:.2f}%")

# Fit scaler on training data ONLY
cicids_scaler = StandardScaler()
X_train_c_scaled = cicids_scaler.fit_transform(X_train_c)
X_test_c_scaled  = cicids_scaler.transform(X_test_c)

print(f"\nScaling done — fitted on training data only")

# Save scaler
joblib.dump(cicids_scaler, '../data/final/cicids_scaler.pkl')
print("Saved → data/final/cicids_scaler.pkl")

Train set: (136292, 32)
Test set:  (34074, 32)

Train label distribution:
label_binary
0    134548
1      1744
Name: count, dtype: int64

Test label distribution:
label_binary
0    33638
1      436
Name: count, dtype: int64

Train attack rate: 1.28%
Test attack rate:  1.28%

Scaling done — fitted on training data only
Saved → data/final/cicids_scaler.pkl


In [8]:
import os

files = [
    '../data/final/train.csv',
    '../data/final/test.csv',
    '../data/final/scaler.pkl',
    '../data/final/feature_names.txt',
    '../data/final/cicids_features.csv',
    '../data/final/cicids_scaler.pkl',
    '../data/cleaned/csic_cleaned_final.csv',
]

print("File check:")
for f in files:
    exists = os.path.exists(f)
    if exists:
        size = os.path.getsize(f)/1024
        print(f"  OK      {f}  ({size:.1f} KB)")
    else:
        print(f"  MISSING {f}")

File check:
  OK      ../data/final/train.csv  (38788.2 KB)
  OK      ../data/final/test.csv  (9697.5 KB)
  OK      ../data/final/scaler.pkl  (3.0 KB)
  OK      ../data/final/feature_names.txt  (0.9 KB)
  MISSING ../data/final/cicids_features.csv
  OK      ../data/final/cicids_scaler.pkl  (2.2 KB)
  OK      ../data/cleaned/csic_cleaned_final.csv  (18170.3 KB)


In [9]:
# Save final CICIDS feature matrix
cicids_final = X_cicids.copy()
cicids_final['label'] = y_cicids.values
cicids_final['attack_type'] = y_cicids_multiclass.values

cicids_final.to_csv('../data/final/cicids_features.csv', index=False)

print(f"Saved → data/final/cicids_features.csv")
print(f"  Shape: {cicids_final.shape}")
print(f"  Features: {len(available)}")
print(f"  Normal:  {(cicids_final['label']==0).sum():,}")
print(f"  Attack:  {(cicids_final['label']==1).sum():,}")
print(f"  Attack types: {cicids_final['attack_type'].value_counts().to_dict()}")

Saved → data/final/cicids_features.csv
  Shape: (170366, 34)
  Features: 32
  Normal:  168,186
  Attack:  2,180
  Attack types: {'BENIGN': 168186, 'Web Attack - Brute Force': 1507, 'Web Attack - XSS': 652, 'Web Attack - SQL Injection': 21}
